# Boosting

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
import pickle

## Definir get_metrics

>Definimos la funcion get_classifier_metrics para analizar los datos que tenemos con los modelos

In [3]:
def get_classifier_metrics(y_predict_test, y_test, y_predict_train, y_train, average='micro'):
    metrics_train = (accuracy_score(y_train, y_predict_train),
                     f1_score(y_train, y_predict_train, average=average),
                     precision_score(y_train, y_predict_train, average=average),
                     recall_score(y_train, y_predict_train, average=average))
    metrics_test = (accuracy_score(y_test, y_predict_test),
                    f1_score(y_test, y_predict_test, average=average),
                    precision_score(y_test, y_predict_test, average=average),
                    recall_score(y_test, y_predict_test, average=average))
    return pd.DataFrame(data=[metrics_train, metrics_test],
                        columns=['Accuracy', 'F1 Score', 'Precision', 'Recall'],
                        index=['Train set', 'Test set'])

## Recolección de Datos

>Los DataFrames completos

In [4]:
with open('/workspaces/adamcn10-intro-ml/data/processed/decision-tree-df-nonull.pkl', 'rb') as file:
    df_nonull = pickle.load(file)

with open('/workspaces/adamcn10-intro-ml/data/processed/decision-tree-df-most-frequent.pkl', 'rb') as file:
    df_most_freq = pickle.load(file)

with open('/workspaces/adamcn10-intro-ml/data/processed/decision-tree-df-knn.pkl', 'rb') as file:
    df_knn = pickle.load(file)

>Datos eliminando valores nulos

In [5]:
X_test_nonull = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-X-test-nonull.csv', 
                            index_col=0)
X_train_nonull = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-X-train-nonull.csv', 
                            index_col=0)
y_test_nonull = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-y-test-nonull.csv', 
                            index_col=0)
y_train_nonull = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-y-train-nonull.csv', 
                            index_col=0)

>Datos sustituyendo 0 por la moda

In [6]:
X_test_most_freq = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-X-test-most-frequent.csv',
                            index_col=0)
X_train_most_freq = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-X-train-most-frequent.csv', 
                            index_col=0)
y_test_most_freq = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-y-test-most-frequent.csv', 
                            index_col=0)
y_train_most_freq = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-y-train-most-frequent.csv', 
                            index_col=0)

>Datos aplicando KNN Imputer

In [7]:
X_test_knn = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-X-test-knn.csv', 
                            index_col=0)
X_train_knn = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-X-train-knn.csv', 
                            index_col=0)
y_test_knn = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-y-test-knn.csv', 
                            index_col=0)
y_train_knn = pd.read_csv('/workspaces/adamcn10-intro-ml/data/edas/decission-tree-y-train-knn.csv', 
                            index_col=0)

>Datos sin solucionar nulos

>Inicializo los datos desde cero, sustituyo los 0 no validos por nan y hago el split para trabajar con ello

In [8]:
df = pd.read_csv('/workspaces/adamcn10-intro-ml/data/raw/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
df[['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']] = df[['Glucose', 
                                                                          'BloodPressure', 
                                                                          'SkinThickness', 
                                                                          'Insulin', 
                                                                          'BMI']].replace(0, np.nan)
df.isnull().sum().sort_values(ascending=False)

Insulin                     374
SkinThickness               227
BloodPressure                35
BMI                          11
Glucose                       5
Pregnancies                   0
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

In [10]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=25)

In [11]:
X_train.isnull().sum().sort_values(ascending=False)

Insulin                     298
SkinThickness               177
BloodPressure                26
BMI                           8
Glucose                       4
Pregnancies                   0
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

## Modelados

### Modelo eliminando 0s

In [12]:
model_nonull = XGBClassifier(random_state = 25)
model_nonull.fit(X_train_nonull, y_train_nonull)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [13]:
y_pred_train_nonull = model_nonull.predict(X_train_nonull)
y_pred_test_nonull = model_nonull.predict(X_test_nonull)
pd.DataFrame(y_pred_test_nonull)

,0
0,0
1,1
2,1
3,0
4,0
...,...
74,1
75,0
76,1
77,1


In [14]:
y_test_nonull

,Outcome
243,1
364,0
469,0
692,0
478,0
...,...
539,1
122,0
689,1
285,0


In [15]:
report_nonull = classification_report(y_test_nonull, y_pred_test_nonull)
print(report_nonull)

              precision    recall  f1-score   support

           0       0.81      0.83      0.82        52
           1       0.65      0.63      0.64        27

    accuracy                           0.76        79
   macro avg       0.73      0.73      0.73        79
weighted avg       0.76      0.76      0.76        79



In [16]:
get_classifier_metrics(y_pred_test_nonull, y_test_nonull, y_pred_train_nonull, y_train_nonull)

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.759494,0.759494,0.759494,0.759494


### Modelo sustituyendo por moda

In [17]:
model_most_freq = XGBClassifier(random_state = 25)
model_most_freq.fit(X_train_most_freq, y_train_most_freq)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [18]:
y_pred_train_most_freq = model_most_freq.predict(X_train_most_freq)
y_pred_test_most_freq = model_most_freq.predict(X_test_most_freq)
pd.DataFrame(y_pred_test_most_freq)

,0
0,0
1,0
2,0
3,1
4,0
...,...
149,0
150,1
151,1
152,1


In [19]:
y_test_most_freq

,Outcome
459,0
39,1
344,0
84,1
700,0
...,...
410,0
114,1
246,0
506,1


In [20]:
report_most_freq = classification_report(y_test_most_freq, y_pred_test_most_freq)
print(report_most_freq)

              precision    recall  f1-score   support

           0       0.82      0.82      0.82       103
           1       0.63      0.63      0.63        51

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154



In [21]:
get_classifier_metrics(y_pred_test_most_freq, y_test_most_freq, y_pred_train_most_freq, y_train_most_freq)

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.753247,0.753247,0.753247,0.753247


### Modelo aplicando KNN 

In [22]:
model_knn = XGBClassifier(random_state = 25)
model_knn.fit(X_train_knn, y_train_knn)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [23]:
y_pred_train_knn = model_knn.predict(X_train_knn)
y_pred_test_knn = model_knn.predict(X_test_knn)
pd.DataFrame(y_pred_test_knn)

,0
0,0
1,0
2,0
3,1
4,0
...,...
149,0
150,1
151,1
152,1


In [24]:
y_test_knn

,Outcome
459,0
39,1
344,0
84,1
700,0
...,...
410,0
114,1
246,0
506,1


In [25]:
report_knn = classification_report(y_test_knn, y_pred_test_knn)
print(report_knn)

              precision    recall  f1-score   support

           0       0.82      0.82      0.82       103
           1       0.63      0.63      0.63        51

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154



In [26]:
get_classifier_metrics(y_pred_test_knn, y_test_knn, y_pred_train_knn, y_train_knn)

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.753247,0.753247,0.753247,0.753247


### Sin tratamiento de datos

In [27]:
model = XGBClassifier(random_state = 25)
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [28]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
pd.DataFrame(y_pred_test)

,0
0,0
1,0
2,0
3,1
4,0
...,...
149,0
150,1
151,1
152,1


In [29]:
y_test

459    0
39     1
344    0
84     1
700    0
      ..
410    0
114    1
246    0
506    1
597    0
Name: Outcome, Length: 154, dtype: int64

In [30]:
report = classification_report(y_test, y_pred_test)
print(report)

              precision    recall  f1-score   support

           0       0.84      0.84      0.84       103
           1       0.69      0.69      0.69        51

    accuracy                           0.79       154
   macro avg       0.77      0.77      0.77       154
weighted avg       0.79      0.79      0.79       154



In [31]:
get_classifier_metrics(y_pred_test, y_test, y_pred_train, y_train)

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.792208,0.792208,0.792208,0.792208


## Optimización de modelos

>Vamos a utilizar GridCV con el scoring recall ya que es el que nos interesa para la temática médica que tenemos en este proyecto para optimizar los tres modelos

In [32]:
hyperparams = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    'scale_pos_weight': [0, 1, -1]
}

### Modelo eliminando 0s

In [33]:
grid_nonull = GridSearchCV(model_nonull, hyperparams, scoring = "recall", cv = 10)
grid_nonull

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...], ...}"
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [34]:
grid_nonull.fit(X_train_nonull, y_train_nonull)

grid_nonull.best_params_

/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [12:59:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [12:59:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [12:59:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [12:59:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iterati

{'max_depth': 5,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100,
 'scale_pos_weight': 1}

In [35]:
grid_nonull.best_estimator_

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [36]:
best_model_nonull = grid_nonull.best_estimator_
y_pred_best_nonull = best_model_nonull.predict(X_test_nonull)
grid_report_nonull = classification_report(y_test_nonull, y_pred_best_nonull)

print(report_nonull, grid_report_nonull)


              precision    recall  f1-score   support

           0       0.81      0.83      0.82        52
           1       0.65      0.63      0.64        27

    accuracy                           0.76        79
   macro avg       0.73      0.73      0.73        79
weighted avg       0.76      0.76      0.76        79
               precision    recall  f1-score   support

           0       0.82      0.87      0.84        52
           1       0.71      0.63      0.67        27

    accuracy                           0.78        79
   macro avg       0.76      0.75      0.75        79
weighted avg       0.78      0.78      0.78        79



### Modelo sustituyendo por moda

In [37]:
grid_most_freq = GridSearchCV(model_most_freq, hyperparams, scoring = "recall", cv = 10)
grid_most_freq

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...], ...}"
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [38]:
grid_most_freq.fit(X_train_most_freq, y_train_most_freq)

grid_most_freq.best_params_

/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:00:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:00:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:00:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:00:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:00:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iterati

{'max_depth': 5,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 50,
 'scale_pos_weight': 1}

In [39]:
grid_most_freq.best_estimator_

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [40]:
best_model_most_freq = grid_most_freq.best_estimator_
y_pred_best_most_freq = best_model_most_freq.predict(X_test_most_freq)
grid_report_most_freq = classification_report(y_test_most_freq, y_pred_best_most_freq)

print(report_most_freq, grid_report_most_freq)

              precision    recall  f1-score   support

           0       0.82      0.82      0.82       103
           1       0.63      0.63      0.63        51

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154
               precision    recall  f1-score   support

           0       0.84      0.83      0.84       103
           1       0.67      0.69      0.68        51

    accuracy                           0.79       154
   macro avg       0.76      0.76      0.76       154
weighted avg       0.79      0.79      0.79       154



### Modelo aplicando KNN 

In [41]:
grid_knn = GridSearchCV(model_knn, hyperparams, scoring = "recall", cv = 10)
grid_knn

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...], ...}"
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [42]:
grid_knn.fit(X_train_knn, y_train_knn)

grid_knn.best_params_

/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:02:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:02:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:02:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:02:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:02:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:02:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iterati

{'max_depth': 5,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 50,
 'scale_pos_weight': 1}

In [43]:
grid_knn.best_estimator_

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [44]:
best_model_knn = grid_knn.best_estimator_
y_pred_best_knn = best_model_knn.predict(X_test_knn)
grid_report_knn = classification_report(y_test_knn, y_pred_best_knn)

print(report_knn, grid_report_knn)

              precision    recall  f1-score   support

           0       0.82      0.82      0.82       103
           1       0.63      0.63      0.63        51

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.75      0.75      0.75       154
               precision    recall  f1-score   support

           0       0.84      0.83      0.84       103
           1       0.67      0.69      0.68        51

    accuracy                           0.79       154
   macro avg       0.76      0.76      0.76       154
weighted avg       0.79      0.79      0.79       154



### modelo sin tratar nulos

In [45]:
grid = GridSearchCV(model, hyperparams, scoring = "recall", cv = 10)
grid

,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...], ...}"
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [46]:
grid.fit(X_train, y_train)

grid.best_params_

/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:03:56] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:03:56] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:03:56] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:03:56] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [13:03:56] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  bst.update(dtrain, iterati

{'max_depth': None,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 50,
 'scale_pos_weight': 1}

In [47]:
grid_knn.best_estimator_

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [48]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
grid_report = classification_report(y_test, y_pred_best)

print(report, grid_report)

              precision    recall  f1-score   support

           0       0.84      0.84      0.84       103
           1       0.69      0.69      0.69        51

    accuracy                           0.79       154
   macro avg       0.77      0.77      0.77       154
weighted avg       0.79      0.79      0.79       154
               precision    recall  f1-score   support

           0       0.83      0.85      0.84       103
           1       0.69      0.65      0.67        51

    accuracy                           0.79       154
   macro avg       0.76      0.75      0.75       154
weighted avg       0.78      0.79      0.78       154



### Valoración de modelos

>Dado que estamos ante un tema médico, la métrica que más nos interesa es la métrica recall, que analiza los verdaderos positivos entre verdaderos positivos y falsos negativos, vamos a compararlos todos, de los 3 modelos básicos y los 3 optimizados para decidir con cual nos quedamos al final

In [49]:
results = {}

results['nonull_recall'] = recall_score(y_test_nonull, y_pred_test_nonull)
results['most_freq_recall'] = recall_score(y_test_most_freq, y_pred_test_most_freq)
results['knn_recall'] = recall_score(y_test_knn, y_pred_test_knn)
results['nonull_grid_recall'] = recall_score(y_test_nonull, y_pred_best_nonull)
results['most_freq_grid_recall'] = recall_score(y_test_most_freq, y_pred_best_most_freq)
results['knn_grid_recall'] = recall_score(y_test_knn, y_pred_best_knn)
results['df_recall'] = recall_score(y_test, y_pred_test)
results['grid_recall'] = recall_score(y_test, y_pred_best)


results

{'nonull_recall': 0.6296296296296297,
 'most_freq_recall': 0.6274509803921569,
 'knn_recall': 0.6274509803921569,
 'nonull_grid_recall': 0.6296296296296297,
 'most_freq_grid_recall': 0.6862745098039216,
 'knn_grid_recall': 0.6862745098039216,
 'df_recall': 0.6862745098039216,
 'grid_recall': 0.6470588235294118}

> ### Grafica comparativa de Sensibilidad de los modelos
>|Modelo     |Eliminando nulos  |Sustituyendo por moda|k-nearest neighbors|Sin tratar nulos  |
>|-----------|------------------|---------------------|-------------------|------------------|
>|Modelo base|0.6296296296296297|0.6274509803921569   |0.6274509803921569 |0.6862745098039216|
>|GridCV     |0.6296296296296297|0.6862745098039216   |0.6862745098039216 |0.6470588235294118|

>### Observaciones - Conclusiones
> El modelo eliminando las filas con nulos me da el mismo resultado siendo optimizado con gridCV y sin serlo y este coincide con el modelo optimizado con GridCV tanto de random forest como de decission tree (siendo este peor que el básico con random forest). Respecto a los sustituidos por la moda y aplicando KNN me dan el mismo resultado básico que los dos tipos de modelo anteriores y optimizados con GridCV empeoran significativamente a los anteriores (Posiblemente también a falta de poner las mejores opciones de hyperparametros en la mejora) y el nuevo modo disponible sin tratar nulos, de manera básica funciona igual que los de la moda y aplicando KNN mejorados pero al aplicar GridCV empeora, sin llegar a acercarse ninguno de ellos al buen resultado que me dieron tanto el decission tree como el random forest en moda y Knn mejorados con GridCV de más de 0.8.

## Guardado de modelos

>Guardaré igualmente los 8 modelos en models

In [50]:
with open('/workspaces/adamcn10-intro-ml/models/xgboost-nonull.pkl', 'wb') as file:
    pickle.dump(model_nonull, file)     

with open('/workspaces/adamcn10-intro-ml/models/xgboost-most-freq.pkl', 'wb') as file:
    pickle.dump(model_most_freq, file)     

with open('/workspaces/adamcn10-intro-ml/models/xgboost-knn.pkl', 'wb') as file:
    pickle.dump(model_knn, file)     

with open('/workspaces/adamcn10-intro-ml/models/xgboost-nonull-grid.pkl', 'wb') as file:
    pickle.dump(best_model_nonull, file)     

with open('/workspaces/adamcn10-intro-ml/models/xgboost-most-freq-grid.pkl', 'wb') as file:
    pickle.dump(best_model_most_freq, file)     

with open('/workspaces/adamcn10-intro-ml/models/xgboost-knn-grid.pkl', 'wb') as file:
    pickle.dump(best_model_knn, file)

with open('/workspaces/adamcn10-intro-ml/models/xgboost.pkl', 'wb') as file:
    pickle.dump(model, file)   

with open('/workspaces/adamcn10-intro-ml/models/xgboost-grid.pkl', 'wb') as file:
    pickle.dump(best_model, file)